# Fase 3b — Chunking: comparación de las cuatro estrategias
Toma el corpus limpio en Markdown de la Fase 2 (`corpus_rag_final_gemini.json`) y lo divide en *chunks* aplicando y comparando **cuatro estrategias** distintas:

- **Estrategia A — Tamaño fijo con solapamiento** (`estrategia_a_fixed_overlap`): `RecursiveCharacterTextSplitter` clásico (chunks de 1000 caracteres, solapamiento de 200).
- **Estrategia B — Por secciones Markdown** (`estrategia_b_markdown`): trocea respetando los encabezados `##`/`###` generados en la Fase 2, subdividiendo recursivamente las secciones que superan el tamaño máximo.
- **Estrategia C — Semántica** (`estrategia_c_semantica`): usa `SemanticChunker` con embeddings multilingües (`paraphrase-multilingual-MiniLM-L12-v2`) para cortar por cambios de significado, con el splitter de tamaño fijo como mecanismo de seguridad.
- **Estrategia D — Jerárquica** (`estrategia_d_markdown_jerarquico`): como la B, pero conserva tanto la sección completa («padre», para dar contexto al LLM) como sus sub-chunks («hijo», para la búsqueda vectorial), enlazados mediante `parent_id`.

Cada chunk incorpora en sus metadatos la información Dublin Core del documento (título, autores, fecha, tema), limpiada por `extraer_metadatos_limpios`. `ejecutar_pipeline_chunking` aplica las cuatro estrategias a todo el corpus y guarda cada resultado en su propio JSON (`chunks_estrategia_A/B/C/D_*.json`). Las últimas celdas auditan la granularidad y los metadatos de cada estrategia (incluida la integridad padre-hijo de la D).

In [1]:
!pip install langchain langchain-text-splitters langchain-experimental langchain-huggingface sentence-transformers

INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 103.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 119.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: huggingface-hub
    Found exist

In [2]:
import uuid
from langchain_text_splitters import RecursiveCharacterTextSplitter

def estrategia_a_fixed_overlap(doc_json, doc_metadata_extra, chunk_size=1000, chunk_overlap=200):
    """
    Aplica chunking de tamaño fijo con solapamiento a un documento procesado.

    Args:
        doc_json (dict): El diccionario original con 'page_content' y 'metadata'.
        doc_metadata_extra (dict): Metadatos adicionales a inyectar (autores, fecha, etc.).
        chunk_size (int): Tamaño máximo del chunk en caracteres.
        chunk_overlap (int): Cantidad de caracteres que se solapan entre chunks.

    Returns:
        list: Lista de diccionarios, donde cada uno es un chunk estructurado.
    """

    if chunk_overlap >= chunk_size:
        raise ValueError("chunk_overlap debe ser menor que chunk_size")

    # 1. Extraemos el texto y los metadatos base
    texto = doc_json["page_content"]
    metadatos_base = doc_json["metadata"].copy()

    # 2. Inyectamos los metadatos adicionales del documento (fecha, título, autores)
    metadatos_base.update(doc_metadata_extra)

    # 3. Configuramos el splitter de LangChain
    # Intentará cortar por párrafos (\n\n), luego líneas (\n), luego espacios, y por último letras
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""]
    )

    # 4. Troceamos el texto
    textos_troceados = splitter.split_text(texto)

    chunks_finales = []

    # 5. Iteramos para construir nuestros objetos finales con IDs y metadatos
    for texto_chunk in (textos_troceados):

        # Si el chunk tiene menos de 30 caracteres, lo ignoramos por ser ruido
        if len(texto_chunk.strip()) < 30:
            continue

        # Generamos un ID único universal (UUID)
        chunk_id = str(uuid.uuid4())

        # Creamos los metadatos específicos para ESTE chunk
        metadatos_chunk = metadatos_base.copy()
        metadatos_chunk["chunk_id"] = chunk_id
        metadatos_chunk["chunk_index"] = len(chunks_finales)
        metadatos_chunk["estrategia"] = "A_Fixed_Overlap"

        # Empaquetamos todo
        chunk_empaquetado = {
            "chunk_id": chunk_id,
            "text": texto_chunk,
            "metadata": metadatos_chunk
        }

        chunks_finales.append(chunk_empaquetado)

    return chunks_finales

In [3]:
import uuid
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

def estrategia_b_markdown(doc_json, doc_metadata_extra, chunk_size=1000, chunk_overlap=200):
    """
    Aplica chunking basado en las secciones Markdown (##) del documento.
    Si una sección es demasiado larga, aplica subdivisión recursiva.
    """

    if chunk_overlap >= chunk_size:
        raise ValueError("chunk_overlap debe ser menor que chunk_size")

    texto = doc_json["page_content"]
    metadatos_base = doc_json["metadata"].copy()
    metadatos_base.update(doc_metadata_extra)

    # 1. Configuramos el splitter de Markdown
    # Le decimos que busque los "##" y "###" y asigne claves distintas
    headers_to_split_on = [
        ("##", "seccion"),
        ("###", "subseccion") # <- ¡Añadimos esta línea!
    ]
    markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

    # Troceamos el documento por sus secciones naturales
    md_splits = markdown_splitter.split_text(texto)

    # 2. Configuramos el splitter de respaldo (para secciones gigantes)
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""]
    )

    chunks_finales = []

    # 3. Procesamos cada sección detectada
    for md_chunk in md_splits:

        # Le pasamos el salvavidas: si la sección mide menos de 1000, devolverá 1 solo trozo.
        # Si mide 3000, devolverá unos 3 o 4 trozos.
        sub_chunks = text_splitter.split_text(md_chunk.page_content)

        for sub_texto in sub_chunks:

            # Mantenemos nuestro filtro de ruido anti-basura
            if len(sub_texto.strip()) < 30:
                continue

            chunk_id = str(uuid.uuid4())
            metadatos_chunk = metadatos_base.copy()

            # Inyectamos el nombre de la sección en los metadatos del chunk
            # Construimos encabezado jerárquico si existe
            encabezado = ""

            if "seccion" in md_chunk.metadata:
                metadatos_chunk["seccion"] = md_chunk.metadata["seccion"]
                encabezado += f"## {md_chunk.metadata['seccion']}\n"

            if "subseccion" in md_chunk.metadata:
                metadatos_chunk["subseccion"] = md_chunk.metadata["subseccion"]
                encabezado += f"### {md_chunk.metadata['subseccion']}\n"

            if encabezado:
                texto_final = f"{encabezado}\n{sub_texto}"
            else:
                texto_final = sub_texto

            metadatos_chunk["chunk_id"] = chunk_id
            metadatos_chunk["chunk_index"] = len(chunks_finales)
            metadatos_chunk["estrategia"] = "B_Markdown_Splitter"

            chunks_finales.append({
                "chunk_id": chunk_id,
                "text": texto_final,
                "metadata": metadatos_chunk
            })

    return chunks_finales

In [4]:
import uuid
from langchain_experimental.text_splitter import SemanticChunker
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings # Importación moderna

def estrategia_c_semantica(doc_json, doc_metadata_extra, chunk_size=1000, chunk_overlap=100, embedding_model=None):
    """
    Aplica chunking semántico y utiliza un límite de tamaño recursivo
    como respaldo para evitar chunks que excedan el límite de tokens.
    """

    if chunk_overlap >= chunk_size:
        raise ValueError("chunk_overlap debe ser menor que chunk_size")

    texto = doc_json["page_content"]
    metadatos_base = doc_json["metadata"].copy()
    metadatos_base.update(doc_metadata_extra)

    # 1. Cargamos el modelo de embeddings local y gratuito
    if embedding_model is None:
        embedding_model = HuggingFaceEmbeddings(model_name="paraphrase-multilingual-MiniLM-L12-v2")

    # 2. Configuramos el Splitter Semántico
    semantic_splitter = SemanticChunker(
        embedding_model,
        breakpoint_threshold_type="percentile",
        breakpoint_threshold_amount=90.0
    )

    # Troceamos por significado
    docs_semanticos = semantic_splitter.create_documents([texto])

    # 3. Configuramos nuestro SALVAVIDAS de tamaño
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""]
    )

    chunks_finales = []

    # 4. Procesamos cada chunk semántico
    for doc_chunk in docs_semanticos:

        # Le aplicamos el límite estricto.
        # Si doc_chunk mide 800, sub_chunks tendrá 1 elemento.
        # Si doc_chunk mide 2500, sub_chunks tendrá 3 elementos.
        sub_chunks = text_splitter.split_text(doc_chunk.page_content)

        for sub_texto in sub_chunks:

            # Nuestro filtro anti-basura
            if len(sub_texto.strip()) < 30:
                continue

            chunk_id = str(uuid.uuid4())
            metadatos_chunk = metadatos_base.copy()
            metadatos_chunk["chunk_id"] = chunk_id
            metadatos_chunk["chunk_index"] = len(chunks_finales)
            metadatos_chunk["estrategia"] = "C_Semantic_Hybrid"

            chunks_finales.append({
                "chunk_id": chunk_id,
                "text": sub_texto,
                "metadata": metadatos_chunk
            })

    return chunks_finales

In [5]:
import uuid
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

def estrategia_d_markdown_jerarquico(doc_json, doc_metadata_extra, chunk_size=1000, chunk_overlap=200):
    """
    Aplica chunking jerárquico: extrae secciones completas (Padres) y las subdivide (Hijos).
    """
    if chunk_overlap >= chunk_size:
        raise ValueError("chunk_overlap debe ser menor que chunk_size")

    texto = doc_json["page_content"]
    metadatos_base = doc_json["metadata"].copy()
    metadatos_base.update(doc_metadata_extra)

    headers_to_split_on = [
        ("##", "seccion"),
        ("###", "subseccion")
    ]
    markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
    md_splits = markdown_splitter.split_text(texto)

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""]
    )

    chunks_finales = [] # Aquí irán los hijos (para búsqueda)
    chunks_padre = []   # Aquí irán los padres (para contexto)

    for md_chunk in md_splits:

        if len(md_chunk.page_content.strip()) < 30:
            continue

        # 1. Creamos el identificador del PADRE
        parent_id = str(uuid.uuid4())

        # 2. Guardamos el chunk PADRE (la sección Markdown entera)
        padre_metadata = metadatos_base.copy()
        padre_metadata.update(md_chunk.metadata) # Añadimos info de sección
        padre_metadata["doc_type"] = "padre"
        padre_metadata["parent_id"] = parent_id # Se referencia a sí mismo
        padre_metadata["chunk_index"] = len(chunks_padre)
        padre_metadata["estrategia"] = "D_Jerarquico"

        chunks_padre.append({
            "chunk_id": parent_id,
            "text": md_chunk.page_content,
            "metadata": padre_metadata
        })

        # 3. Troceamos para obtener los HIJOS
        sub_chunks = text_splitter.split_text(md_chunk.page_content)

        for sub_texto in sub_chunks:
            if len(sub_texto.strip()) < 30:
                continue

            chunk_id = str(uuid.uuid4())
            metadatos_chunk = metadatos_base.copy()
            metadatos_chunk.update(md_chunk.metadata)

            # ¡La clave del chunking jerárquico!
            metadatos_chunk["parent_id"] = parent_id
            metadatos_chunk["doc_type"] = "hijo"
            metadatos_chunk["chunk_id"] = chunk_id
            metadatos_chunk["chunk_index"] = len(chunks_finales)
            metadatos_chunk["estrategia"] = "D_Jerarquico"

            # (Opcional) Puedes mantener tu lógica de inyectar el encabezado en el texto
            encabezado = ""
            if "seccion" in md_chunk.metadata:
                encabezado += f"## {md_chunk.metadata['seccion']}\n"
            if "subseccion" in md_chunk.metadata:
                encabezado += f"### {md_chunk.metadata['subseccion']}\n"

            texto_final = f"{encabezado}\n{sub_texto}" if encabezado else sub_texto

            chunks_finales.append({
                "chunk_id": chunk_id,
                "text": texto_final,
                "metadata": metadatos_chunk
            })

    # Devolvemos tanto padres como hijos
    return chunks_padre, chunks_finales

In [6]:
import json
import os
import uuid
from langchain_huggingface import HuggingFaceEmbeddings # Importamos aquí para instanciar una sola vez

# Rutas en Google Colab
CORPUS_FILE = 'corpus_rag_final_gemini.json'
METADATA_DIR = 'metadatos'  # Carpeta donde subirás los JSONs de metadatos

OUTPUT_FILE_A = 'chunks_estrategia_A_fixed.json'
OUTPUT_FILE_B = 'chunks_estrategia_B_markdown.json'
OUTPUT_FILE_C = 'chunks_estrategia_C_semantica.json'
OUTPUT_FILE_D = 'chunks_estrategia_D_jerarquico.json'

def extraer_metadatos_limpios(ruta_json_metadatos):
    """
    Lee el JSON estilo Dublin Core y extrae los campos clave
    con nombres limpios para nuestro sistema RAG.
    """
    if not os.path.exists(ruta_json_metadatos):
        print(f"ATENCIÓN: No se encontró el archivo de metadatos en -> {ruta_json_metadatos}")
        return {} # Devuelve vacío si no encuentra el archivo

    with open(ruta_json_metadatos, 'r', encoding='utf-8') as f:
        meta_raw = json.load(f)

    # Mapeamos los campos de Dublin Core a algo más legible
    meta_limpio = {
        "titulo": meta_raw.get("DC.title", "Título desconocido"),
        "autores": meta_raw.get("DC.creator", []),
        "fecha_publicacion": meta_raw.get("DCTERMS.issued", "Fecha desconocida"),
        "tema": meta_raw.get("DC.subject", "Sin tema")
    }

    # Convertimos listas a strings si es necesario (a algunas BBDD vectoriales no les gustan las listas en metadatos)
    if isinstance(meta_limpio["autores"], list):
        meta_limpio["autores"] = ", ".join(meta_limpio["autores"])

    if isinstance(meta_limpio["tema"], list):
        meta_limpio["tema"] = ", ".join(meta_limpio["tema"])

    return meta_limpio


def ejecutar_pipeline_chunking():
    """
    Orquesta el pipeline completo de chunking: carga el corpus limpio de la
    Fase 2 y, para cada documento, recupera sus metadatos Dublin Core
    (extraer_metadatos_limpios) y aplica las cuatro estrategias de chunking
    (A: tamaño fijo, B: secciones Markdown, C: semántica, D: jerárquica),
    guardando los resultados de cada estrategia en su propio JSON.
    """
    print("Iniciando el Pipeline de Chunking ...")

    print("Cargando modelo de embeddings en memoria...")
    modelo_embeddings_global = HuggingFaceEmbeddings(model_name="paraphrase-multilingual-MiniLM-L12-v2")
    print("Modelo cargado y listo.")

    # 1. Cargamos el corpus completo
    with open(CORPUS_FILE, 'r', encoding='utf-8') as f:
        corpus = json.load(f)

    print(f"Se han cargado {len(corpus)} documentos médicos.")

    # Listas para almacenar TODOS los chunks del corpus separados por estrategia
    todos_chunks_A = []
    todos_chunks_B = []
    todos_chunks_C = []
    todos_chunks_D = []

    # 2. Iteramos sobre cada documento
    for i, doc in enumerate(corpus):

        nombre_archivo_original = doc["metadata"]["source"]

        # Le quitamos la extensión (.pdf) para buscar su .json correspondiente
        nombre_base = os.path.splitext(nombre_archivo_original)[0]
        ruta_meta = os.path.join(METADATA_DIR, f"{nombre_base}.json")

        # Extraemos los metadatos específicos de este documento
        metadatos_extra = extraer_metadatos_limpios(ruta_meta)

        print(f"Procesando [{i+1}/{len(corpus)}]: {metadatos_extra.get('titulo', nombre_archivo_original)}")

        # --- ESTRATEGIA A ---
        chunks_a = estrategia_a_fixed_overlap(doc, metadatos_extra)
        todos_chunks_A.extend(chunks_a)

        # --- ESTRATEGIA B ---
        chunks_b = estrategia_b_markdown(doc, metadatos_extra)
        todos_chunks_B.extend(chunks_b)

        # --- ESTRATEGIA C ---
        # Nota: La primera vez que se ejecute, descargará el modelo paraphrase-multilingual-MiniLM-L12-v2
        chunks_c = estrategia_c_semantica(doc, metadatos_extra, chunk_size=1000, chunk_overlap=100, embedding_model=modelo_embeddings_global)
        todos_chunks_C.extend(chunks_c)

        # --- ESTRATEGIA D (Jerárquica) ---
        # Desempaquetamos los padres y los hijos
        padres_d, hijos_d = estrategia_d_markdown_jerarquico(doc, metadatos_extra)

        # Añadimos ambos al gran listado final
        todos_chunks_D.extend(padres_d)
        todos_chunks_D.extend(hijos_d)

    # 3. Guardamos los resultados en 3 JSONs independientes
    print("\nGuardando resultados...")

    with open(OUTPUT_FILE_A, 'w', encoding='utf-8') as f:
        json.dump(todos_chunks_A, f, ensure_ascii=False, indent=4)

    with open(OUTPUT_FILE_B, 'w', encoding='utf-8') as f:
        json.dump(todos_chunks_B, f, ensure_ascii=False, indent=4)

    with open(OUTPUT_FILE_C, 'w', encoding='utf-8') as f:
        json.dump(todos_chunks_C, f, ensure_ascii=False, indent=4)

    with open(OUTPUT_FILE_D, 'w', encoding='utf-8') as f:
        json.dump(todos_chunks_D, f, ensure_ascii=False, indent=4)

    print("¡Pipeline completado con éxito!")
    print(f"Resumen de chunks generados:")
    print(f"   - Estrategia A (Fixed): {len(todos_chunks_A)} chunks")
    print(f"   - Estrategia B (Markdown): {len(todos_chunks_B)} chunks")
    print(f"   - Estrategia C (Semántica): {len(todos_chunks_C)} chunks")
    print(f"   - Estrategia D (Jerárquica): {len(todos_chunks_D)} chunks")


if __name__ == "__main__":
    ejecutar_pipeline_chunking()

Iniciando el Pipeline de Chunking ...
Cargando modelo de embeddings en memoria...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo cargado y listo.
Se han cargado 20 documentos médicos.
Procesando [1/20]: Guía hospitalaria de terapéutica antibiótica en adultos
Procesando [2/20]: Protocolo regional de prevención y detección de violencia en la mujer mayor de 65 años
Procesando [3/20]: Protocolo Regional para la indicación, uso y autorización de dispensación de medicamentos sujetos a prescripción médica por parte de las/os enfermeras/os: ostomías
Procesando [4/20]: Protocolo para la detección precoz y manejo de casos de mpox en la Región de Murcia
Procesando [5/20]: Protocolo de vacunación antigripal y antineumocócica. Temporada 2019-2020
Procesando [6/20]: Protocolo para la detección y atención de la violencia de género en atención primaria
Procesando [7/20]: Atención al tabaquismo en atención primaria. Guía de práctica clínica
Procesando [8/20]: Protocolo Regional para la indicación, uso y autorización de dispensación de medicamentos sujetos a prescripción médica por parte de las/os enfermeras/os: heridas
Pr

In [7]:
import json
import statistics
import random

ARCHIVOS = {
    "Estrategia A (Fixed)": 'chunks_estrategia_A_fixed.json',
    "Estrategia B (Markdown)": 'chunks_estrategia_B_markdown.json',
    "Estrategia C (Semántica)": 'chunks_estrategia_C_semantica.json',
    "Estrategia D (Jerárquica)": 'chunks_estrategia_D_jerarquico.json'
}

def calcular_metricas(lista_chunks):
    """Función auxiliar para calcular e imprimir métricas de una lista de chunks"""

    longitudes = [len(c["text"]) for c in lista_chunks]
    print(f" -> Total de chunks:          {len(lista_chunks)}")
    print(f" -> Longitud Media:           {statistics.mean(longitudes):.0f} caracteres")
    print(f" -> Longitud Mediana:         {statistics.median(longitudes):.0f} caracteres")
    print(f" -> Longitud Máxima:          {max(longitudes)} caracteres")
    print(f" -> Longitud Mínima:          {min(longitudes)} caracteres")

def auditar_resultados_y_granularidad():
    """
    Carga los chunks generados por cada una de las cuatro estrategias y
    muestra sus métricas de granularidad (calcular_metricas), añadiendo
    para la estrategia jerárquica un análisis específico por nivel
    (padres/hijos) y una validación de integridad de las relaciones
    parent-child, además de un control de calidad de los metadatos base
    y una muestra aleatoria de texto.
    """
    print("INICIANDO AUDITORÍA Y ANÁLISIS DE GRANULARIDAD POST-CHUNKING")
    print("="*75)

    for nombre, archivo in ARCHIVOS.items():
        try:
            with open(archivo, 'r', encoding='utf-8') as f:
                chunks = json.load(f)
        except FileNotFoundError:
            print(f"Archivo no encontrado: {archivo}")
            continue

        if not chunks:
            print(f"{nombre}: El archivo está vacío.")
            continue

        print(f"\n{nombre.upper()}")
        print("-" * 50)
        print("   [ MÉTRICAS DE GRANULARIDAD ]")

        # --- LÓGICA ESPECÍFICA PARA LA ESTRATEGIA JERÁRQUICA ---
        if "Jerárquica" in nombre:
            padres = [c for c in chunks if c.get("metadata", {}).get("doc_type") == "padre"]
            hijos = [c for c in chunks if c.get("metadata", {}).get("doc_type") == "hijo"]

            print("\n   --- Nivel PADRES (Contexto LLM) ---")
            calcular_metricas(padres)

            print("\n   --- Nivel HIJOS (Vectores de Búsqueda) ---")
            calcular_metricas(hijos)

            # --- VALIDACIONES RELACIONALES (Jerarquía) ---
            print("\n   [ CONTROL DE CALIDAD JERÁRQUICO ]")
            ids_padres_reales = set(p["chunk_id"] for p in padres)
            ids_padres_en_hijos = set(h["metadata"].get("parent_id") for h in hijos)

            huerfanos = [h for h in hijos if h["metadata"].get("parent_id") not in ids_padres_reales]

            print(f"   -> Integridad Parent-Child:   {'OK' if not huerfanos else f'ERROR: {len(huerfanos)} hijos sin padre válido'}")

            if padres and hijos:
                ratio = len(hijos) / len(padres)
                print(f"   -> Ratio de ramificación:     {ratio:.1f} hijos por cada padre de media")

        # --- LÓGICA PARA ESTRATEGIAS PLANAS (A, B, C) ---
        else:
            calcular_metricas(chunks)

        # --- VALIDACIONES DE CALIDAD (METADATOS BASE) ---
        print("\n   [ CONTROL DE CALIDAD Y METADATOS ]")
        if "Jerárquica" in nombre:
            # En la jerárquica, auditamos a un hijo (los padres tienen metadatos reducidos)
            chunk_prueba = hijos[0]["metadata"] if hijos else {}
        else:
            chunk_prueba = chunks[0]["metadata"] if chunks else {}

        metadatos_base = ["chunk_id", "chunk_index", "titulo", "autores", "estrategia"]
        metadatos_ok = all(k in chunk_prueba for k in metadatos_base)
        print(f"   -> Integridad metadatos base: {'OK' if metadatos_ok else 'FALTAN CAMPOS'}")

        # Validación específica para Estrategia B (Markdown)
        if "Markdown" in nombre or "Jerárquica" in nombre:
            chunks_con_seccion = sum(1 for c in chunks if "seccion" in c["metadata"] or "subseccion" in c["metadata"])
            porcentaje = (chunks_con_seccion / len(chunks)) * 100 if len(chunks) > 0 else 0
            print(f"   -> Chunks con jerarquía MD:   {chunks_con_seccion} ({porcentaje:.1f}%)")

        # Muestra visual
        muestra = random.choice(chunks)
        tipo_str = f" [{muestra['metadata'].get('doc_type', 'estándar').upper()}]" if "Jerárquica" in nombre else ""
        texto_muestra = muestra['text'].replace('\n', ' ')[:100]
        print(f"   -> Muestra al azar{tipo_str}:  \"{texto_muestra}...\"")
        print("="*75)

if __name__ == "__main__":
    auditar_resultados_y_granularidad()

INICIANDO AUDITORÍA Y ANÁLISIS DE GRANULARIDAD POST-CHUNKING

ESTRATEGIA A (FIXED)
--------------------------------------------------
   [ MÉTRICAS DE GRANULARIDAD ]
 -> Total de chunks:          1654
 -> Longitud Media:           714 caracteres
 -> Longitud Mediana:         784 caracteres
 -> Longitud Máxima:          999 caracteres
 -> Longitud Mínima:          31 caracteres

   [ CONTROL DE CALIDAD Y METADATOS ]
   -> Integridad metadatos base: OK
   -> Muestra al azar:  "## BACTERIURIA ASINTOMÁTICA  | INDICACIONES | ETIOLOGÍA HABITUAL | TRATAMIENTO DE ELECCIÓN | TRATAMI..."

ESTRATEGIA B (MARKDOWN)
--------------------------------------------------
   [ MÉTRICAS DE GRANULARIDAD ]
 -> Total de chunks:          2031
 -> Longitud Media:           587 caracteres
 -> Longitud Mediana:         582 caracteres
 -> Longitud Máxima:          1181 caracteres
 -> Longitud Mínima:          44 caracteres

   [ CONTROL DE CALIDAD Y METADATOS ]
   -> Integridad metadatos base: OK
   -> Chunks con 